## Imports

In [60]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
import shap
import pandas as pd
from sklearn.model_selection import train_test_split

In [61]:
# Load data
diabetes = pd.read_csv("data\diabetes\diabetic_data.csv")
ids = pd.read_csv("data\diabetes\IDS_mapping.csv")

## Preprocessing

In [62]:
diabetes = diabetes.replace({'?': np.nan})
diabetes['readmitted'] = diabetes['readmitted'].replace({'NO': 0, '>30': 0, '<30': 1})
diabetes['diabetesMed'] = diabetes['diabetesMed'].replace({'No': 0, 'Yes': 1})
diabetes['change'] = diabetes['change'].replace({'No': 0, 'Ch': 1})
diabetes = diabetes.drop(columns=['encounter_id', 'patient_nbr'])
# Drop high-missing, messy columns
diabetes = diabetes.drop(columns=['weight', 'payer_code', 'medical_specialty'])
# Deleting rows with invalid gender in EDA we found there are 3
diabetes = diabetes[diabetes["gender"] != "Unknown/Invalid"]
# Replace missing race values with Unknown
diabetes["race"] = diabetes["race"].fillna("Unknown")
# Encoding age groups as numeric values 
age_map = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}

diabetes['age'] = diabetes['age'].map(age_map)

### Groups of interest
sex = diabetes["gender"].values
races = diabetes["race"].values
print(np.unique(sex))
print(np.unique(races))


['Female' 'Male']
['AfricanAmerican' 'Asian' 'Caucasian' 'Hispanic' 'Other' 'Unknown']


In [63]:
diabetes['metformin'].unique()

<StringArray>
['No', 'Steady', 'Up', 'Down']
Length: 4, dtype: str

In [64]:
feature_names = [
    'race',
    'gender',
    'age',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'diag_1',
    'diag_2',
    'diag_3',
    'number_diagnoses',
    'max_glu_serum',
    'A1Cresult',
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'glipizide',
    'glyburide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'insulin',
    'glyburide-metformin',
    'change',
    'diabetesMed'
]

target_name = "readmitted"

In [65]:
diabetes['gender'] = diabetes['gender'].replace({'Male': 1, 'Female': 0})
categorical_cols = [
    'race',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'diag_1',
    'diag_2',
    'diag_3',
    'max_glu_serum',
    'A1Cresult',
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'glipizide',
    'glyburide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'insulin',
    'glyburide-metformin'
]

# One-hot encode categorical variables
diabetes_encoded = pd.get_dummies(
    diabetes,
    columns=categorical_cols,
    drop_first=True,
    dummy_na=False
)

print(diabetes_encoded.shape)
diabetes_encoded.head()

(101763, 2369)


,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,...,acarbose_Up,miglitol_No,miglitol_Steady,miglitol_Up,insulin_No,insulin_Steady,insulin_Up,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up
0,0,5,1,41,0,1,0,0,0,1,...,False,True,False,False,True,False,False,True,False,False
1,0,15,3,59,0,18,0,0,0,9,...,False,True,False,False,False,False,True,True,False,False
2,0,25,2,11,5,13,2,0,1,6,...,False,True,False,False,True,False,False,True,False,False
3,1,35,2,44,1,16,0,0,0,7,...,False,True,False,False,False,False,True,True,False,False
4,1,45,1,51,0,8,0,0,0,5,...,False,True,False,False,False,True,False,True,False,False


In [66]:
train_data, test_data = train_test_split(
    diabetes_encoded,
    test_size=0.2,
    random_state=0,
    stratify=diabetes_encoded["readmitted"]
)

In [ ]:
def get_log_reg_pipeline(
    seed: int = 70766,
    max_iter: int = 5000,
    penalty: str = 'l2',
    C: float = 0.8497534359086438,
    tol: float = 1e-4,
    solver: str = 'saga'
):
    scaler = StandardScaler() # Standard scale for log reg
    model = LogisticRegression(
        penalty = penalty,
        C = C,
        random_state = seed,
        max_iter = max_iter,
        tol = tol,
        solver = solver
    )
    pipeline = Pipeline([('scaler', scaler), ('classifier', model)])
    return pipeline

In [ ]:
# Get pipeline and fit to training data
log_reg = get_log_reg_pipeline()
log_reg.fit(X_train, y_train)

# Predict on test set
log_reg_y_pred = log_reg.predict(X_test)

# Evaluate
print_classification_results(y_test, log_reg_y_pred)